# SkyGuard — Full Pipeline (single self-contained notebook)

This notebook is the **one deliverable notebook** the brief asks for: it runs
top to bottom from nothing but the `data/` folder (plus the small `src/`
helper package that all of this project's notebooks share) and produces the
final `submission.csv`. It doesn't read anything cached by another notebook
run -- every feature panel, model, and prediction here is built fresh in this
run.

It walks the same five stages as the rest of the project, condensed into one
place:

1. **Per-modality baselines** -- tabular / signal / text alone, so there's a
   number to beat before fusing.
2. **Missingness handling** -- sensor-dropout and empty-note handling, plus a
   modality-dropout training augmentation, with an ablation proving what it
   does and doesn't fix.
3. **Mid-level fusion** -- one LightGBM model on tabular+signal+text concatenated.
4. **Multi-task auxiliary head** -- `failure_mode` as a 6-class joint target,
   compared against plain fusion on the same validation folds.
5. **Calibration** -- Isotonic vs. Platt scaling on out-of-fold predictions,
   then the final calibrated test predictions written to `submission.csv`.

All cross-validation throughout is 5-fold `StratifiedGroupKFold` on
`drone_id`, so no aircraft's flights ever split across train/validation.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import IsotonicRegression
from sklearn.metrics import brier_score_loss

from src.data import load_split
from src.cv import get_folds, assert_disjoint_groups
from src.metrics import auprc, report_folds
from src.features_tabular import build_tabular_features
from src.features_signal import build_signal_features
from src.features_text import missing_indicator, tfidf_features, embed_notes
from src.feature_panel import build_feature_panel
from src.modality_dropout import apply_modality_dropout

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

LGB_PARAMS = dict(
    objective="binary",
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=-1,
)


## 0. Load data and build CV folds

In [2]:
train = load_split("train")
tab, notes, signals, channel_names = train["tab"], train["notes"], train["signals"], train["channel_names"]
note_text = notes["maintenance_note"]
y = tab["failure_within_horizon"].values
m = tab["failure_mode"].astype(str).str.strip().values
print(f"Train: {len(tab)} flights, {tab['drone_id'].nunique()} drones, base rate = {y.mean():.4f}")

test = load_split("test")
tab_test, notes_test, signals_test = test["tab"], test["notes"], test["signals"]
note_text_test = notes_test["maintenance_note"]
print(f"Test: {len(tab_test)} flights, {tab_test['drone_id'].nunique()} drones")

folds = list(get_folds(tab, n_splits=5, seed=RANDOM_STATE))
for tr_idx, val_idx in folds:
    assert_disjoint_groups(tab, tr_idx, val_idx)
print(f"{len(folds)} group-disjoint folds ready")


Train: 15872 flights, 620 drones, base rate = 0.1251


Test: 7534 flights, 290 drones
5 group-disjoint folds ready


## 1. Stage 1 — per-modality baselines

What each modality is worth alone, before any fusion. This is the number
fusion has to clear to prove the modalities actually interact.


In [3]:
# --- Tabular ---
X_tab, tab_cat_cols = build_tabular_features(tab)
tab_feature_cols = [c for c in X_tab.columns if c != "flight_id"]

tab_oof = np.zeros(len(tab))
tab_fold_scores = []
for tr_idx, val_idx in folds:
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_tab.loc[tr_idx, tab_feature_cols], y[tr_idx], categorical_feature=tab_cat_cols)
    val_pred = model.predict_proba(X_tab.loc[val_idx, tab_feature_cols])[:, 1]
    tab_oof[val_idx] = val_pred
    tab_fold_scores.append(auprc(y[val_idx], val_pred))
tab_results = report_folds(tab_fold_scores, "Tabular")


Tabular: fold AUPRCs = [0.2941, 0.2831, 0.2759, 0.3152, 0.2995]
Tabular: mean=0.2936  std=0.0136  (floor~0.125)


In [4]:
# --- Signal (hand-engineered spectral/statistical features) ---
X_sig = build_signal_features(signals, channel_names)
sig_feature_cols = X_sig.columns.tolist()

sig_oof = np.zeros(len(tab))
sig_fold_scores = []
for tr_idx, val_idx in folds:
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_sig.loc[tr_idx, sig_feature_cols], y[tr_idx])
    val_pred = model.predict_proba(X_sig.loc[val_idx, sig_feature_cols])[:, 1]
    sig_oof[val_idx] = val_pred
    sig_fold_scores.append(auprc(y[val_idx], val_pred))
sig_results = report_folds(sig_fold_scores, "Signal")


Signal: fold AUPRCs = [0.2981, 0.2788, 0.2795, 0.2839, 0.3037]
Signal: mean=0.2888  std=0.0102  (floor~0.125)


In [5]:
# --- Text: TF-IDF (fit inside each fold) ---
miss = missing_indicator(note_text).reshape(-1, 1)
tfidf_oof = np.zeros(len(tab))
tfidf_fold_scores = []
for tr_idx, val_idx in folds:
    tr_X, val_X = tfidf_features(note_text.iloc[tr_idx], note_text.iloc[val_idx])
    tr_X, val_X = hstack([tr_X, miss[tr_idx]]), hstack([val_X, miss[val_idx]])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(tr_X, y[tr_idx])
    val_pred = clf.predict_proba(val_X)[:, 1]
    tfidf_oof[val_idx] = val_pred
    tfidf_fold_scores.append(auprc(y[val_idx], val_pred))
tfidf_results = report_folds(tfidf_fold_scores, "Text (TF-IDF)")


Text (TF-IDF): fold AUPRCs = [0.4394, 0.4106, 0.4207, 0.4178, 0.4261]
Text (TF-IDF): mean=0.4229  std=0.0096  (floor~0.125)


In [6]:
# --- Text: frozen sentence embeddings (only 24 unique note templates in the whole dataset) ---
unique_texts = note_text.unique()
emb_lookup = dict(zip(unique_texts, embed_notes(unique_texts.tolist())))
note_emb = np.vstack(note_text.map(emb_lookup).values)

emb_oof = np.zeros(len(tab))
emb_fold_scores = []
for tr_idx, val_idx in folds:
    tr_X = np.hstack([note_emb[tr_idx], miss[tr_idx]])
    val_X = np.hstack([note_emb[val_idx], miss[val_idx]])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(tr_X, y[tr_idx])
    val_pred = clf.predict_proba(val_X)[:, 1]
    emb_oof[val_idx] = val_pred
    emb_fold_scores.append(auprc(y[val_idx], val_pred))
emb_results = report_folds(emb_fold_scores, "Text (BGE embeddings)")


C:\Users\aneja\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Text (BGE embeddings): fold AUPRCs = [0.4399, 0.4084, 0.4253, 0.4159, 0.4261]
Text (BGE embeddings): mean=0.4231  std=0.0106  (floor~0.125)


In [7]:
stage1_summary = pd.DataFrame([
    {"modality": "Tabular", "mean_AUPRC": tab_results["mean"], "std": tab_results["std"]},
    {"modality": "Signal (hand-engineered)", "mean_AUPRC": sig_results["mean"], "std": sig_results["std"]},
    {"modality": "Text (TF-IDF)", "mean_AUPRC": tfidf_results["mean"], "std": tfidf_results["std"]},
    {"modality": "Text (BGE embeddings)", "mean_AUPRC": emb_results["mean"], "std": emb_results["std"]},
]).set_index("modality")
stage1_summary


,mean_AUPRC,std
modality,,
Tabular,0.293566,0.013593
Signal (hand-engineered),0.288801,0.010202
Text (TF-IDF),0.422911,0.009633
Text (BGE embeddings),0.423146,0.010598


## 2. Stage 2 — missingness handling

Build the combined tabular+signal+text-embedding feature panel that every
later stage reuses, then prove modality-dropout training augmentation
actually changes model behavior under simulated missingness. Only `signal`
and `text` are droppable -- tabular has zero missingness anywhere in this
dataset, so there's no real condition synthetic tabular dropout would model.


In [8]:
panel_clean, cat_cols, text_cols = build_feature_panel(tab, note_text, signals, channel_names)
feature_cols = [c for c in panel_clean.columns if c != "flight_id"]
panel_test, _, _ = build_feature_panel(tab_test, note_text_test, signals_test, channel_names)
print("train panel:", panel_clean.shape, " test panel:", panel_test.shape)

# globally-degraded panels, used only for *evaluating* robustness, never for training
all_blank_text = pd.Series([""] * len(note_text), index=note_text.index)
panel_text_blank, _, _ = build_feature_panel(tab, all_blank_text, signals, channel_names)
all_blank_signal = np.zeros_like(signals)
panel_signal_blank, _, _ = build_feature_panel(tab, note_text, all_blank_signal, channel_names)


train panel: (15872, 1181)  test panel: (7534, 1181)


In [9]:
ablation_rows = []
for fold_i, (tr_idx, val_idx) in enumerate(folds):
    model_a = lgb.LGBMClassifier(**LGB_PARAMS)
    model_a.fit(panel_clean.loc[tr_idx, feature_cols], y[tr_idx], categorical_feature=cat_cols)

    sig_aug, txt_aug, _, _ = apply_modality_dropout(signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i)
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)
    model_b = lgb.LGBMClassifier(**LGB_PARAMS)
    model_b.fit(panel_aug.loc[tr_idx, feature_cols], y[tr_idx], categorical_feature=cat_cols)

    for name, model in [("A (no dropout aug)", model_a), ("B (with dropout aug)", model_b)]:
        normal = auprc(y[val_idx], model.predict_proba(panel_clean.loc[val_idx, feature_cols])[:, 1])
        text_missing = auprc(y[val_idx], model.predict_proba(panel_text_blank.loc[val_idx, feature_cols])[:, 1])
        signal_missing = auprc(y[val_idx], model.predict_proba(panel_signal_blank.loc[val_idx, feature_cols])[:, 1])
        ablation_rows.append({"fold": fold_i, "model": name, "normal": normal, "text_missing": text_missing, "signal_missing": signal_missing})

ablation = pd.DataFrame(ablation_rows)
ablation.groupby("model")[["normal", "text_missing", "signal_missing"]].mean()


,normal,text_missing,signal_missing
model,,,
A (no dropout aug),0.522395,0.308319,0.491025
B (with dropout aug),0.518727,0.306357,0.519057


## 3. Stage 3 — mid-level fusion

One LightGBM model on the concatenated tabular+signal+text panel, trained
with the modality-dropout augmentation from Stage 2 on each fold's training
rows. This is the model every fusion-stage number is compared against.


In [10]:
oof_fusion = np.zeros(len(tab))
test_preds_fusion_folds = []
fusion_fold_scores = []

for fold_i, (tr_idx, val_idx) in enumerate(folds):
    sig_aug, txt_aug, _, _ = apply_modality_dropout(signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i)
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)

    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(panel_aug.loc[tr_idx, feature_cols], y[tr_idx], categorical_feature=cat_cols)

    val_pred = model.predict_proba(panel_clean.loc[val_idx, feature_cols])[:, 1]
    oof_fusion[val_idx] = val_pred
    fusion_fold_scores.append(auprc(y[val_idx], val_pred))
    test_preds_fusion_folds.append(model.predict_proba(panel_test[feature_cols])[:, 1])

fusion_results = report_folds(fusion_fold_scores, "Mid-fusion")
test_preds_fusion = np.mean(test_preds_fusion_folds, axis=0)


Mid-fusion: fold AUPRCs = [0.5254, 0.5018, 0.507, 0.5343, 0.5252]
Mid-fusion: mean=0.5187  std=0.0123  (floor~0.125)


## 4. Stage 4 — multi-task auxiliary head (`failure_mode`)

`failure_mode` (bearing/battery/none) is train-only, so it can't be a model
*input* -- but it can be an auxiliary *target*. Reformulated here as a joint
6-class problem over `(failure_within_horizon, failure_mode)`, marginalizing
back to `P(failure_within_horizon=1)` at inference by summing the three
"fails" classes. Same CV folds, same dropout augmentation, same panel, so
this is a fair apples-to-apples comparison against Stage 3.


In [11]:
def to_joint_class(y_val, m_val):
    base = {"none": 0, "battery": 1, "bearing": 2}[m_val]
    return base + (3 if y_val == 1 else 0)

joint_y = np.array([to_joint_class(yv, mv) for yv, mv in zip(y, m)])
pd.Series(joint_y).value_counts().sort_index()


0    13817
1       23
2       46
3      575
4      452
5      959
Name: count, dtype: int64

In [12]:
MULTITASK_PARAMS = {**LGB_PARAMS, "objective": "multiclass", "num_class": 6}

oof_multitask = np.zeros(len(tab))
test_preds_multitask_folds = []
multitask_fold_scores = []

for fold_i, (tr_idx, val_idx) in enumerate(folds):
    sig_aug, txt_aug, _, _ = apply_modality_dropout(signals, note_text, tr_idx, p_signal=0.15, p_text=0.15, seed=RANDOM_STATE + fold_i)
    panel_aug, _, _ = build_feature_panel(tab, txt_aug, sig_aug, channel_names)

    model = lgb.LGBMClassifier(**MULTITASK_PARAMS)
    model.fit(panel_aug.loc[tr_idx, feature_cols], joint_y[tr_idx], categorical_feature=cat_cols)

    val_probs = model.predict_proba(panel_clean.loc[val_idx, feature_cols])
    val_pred = val_probs[:, 3] + val_probs[:, 4] + val_probs[:, 5]
    oof_multitask[val_idx] = val_pred
    multitask_fold_scores.append(auprc(y[val_idx], val_pred))

    test_probs = model.predict_proba(panel_test[feature_cols])
    test_preds_multitask_folds.append(test_probs[:, 3] + test_probs[:, 4] + test_probs[:, 5])

multitask_results = report_folds(multitask_fold_scores, "Multi-task (6-class)")
test_preds_multitask = np.mean(test_preds_multitask_folds, axis=0)


Multi-task (6-class): fold AUPRCs = [0.512, 0.5035, 0.4973, 0.5293, 0.5233]
Multi-task (6-class): mean=0.5131  std=0.0119  (floor~0.125)


In [13]:
stage34_summary = pd.DataFrame([
    {"model": "Stage 3: mid-fusion", "mean_AUPRC": fusion_results["mean"], "std": fusion_results["std"], "pooled_OOF": auprc(y, oof_fusion)},
    {"model": "Stage 4: multi-task", "mean_AUPRC": multitask_results["mean"], "std": multitask_results["std"], "pooled_OOF": auprc(y, oof_multitask)},
]).set_index("model")
stage34_summary


,mean_AUPRC,std,pooled_OOF
model,,,
Stage 3: mid-fusion,0.518727,0.012273,0.516346
Stage 4: multi-task,0.513074,0.011927,0.512357


## 5. Stage 5 — model selection and calibration

**Model selection first.** Stage 4's auxiliary head was meant to regularize
the representation, but it scores *lower* than plain Stage 3 fusion on
validated AUPRC, consistently across repeated runs (~0.005-0.006 mean AUPRC
below Stage 3, bigger than run-to-run noise). Since AUPRC is the literal
competition metric, the final model is **Stage 3's plain fusion**, not the
multi-task version -- the auxiliary-head experiment was worth running and
worth keeping in this notebook as a documented negative result, but it isn't
what gets calibrated and submitted.

**Then calibration**, fit on Stage 3's pooled out-of-fold predictions.
Isotonic regression improves Brier score the most but introduces flat-bin
ties that cost a little ranking resolution; Platt scaling (a monotonic
sigmoid) can't lose any AUPRC by construction, since it never reorders
predictions. We pick whichever keeps AUPRC intact, since that's the
competition metric.


In [14]:
brier_raw = brier_score_loss(y, oof_fusion)
auprc_raw = auprc(y, oof_fusion)

isotonic = IsotonicRegression(out_of_bounds="clip")
isotonic.fit(oof_fusion, y)
cal_oof_iso = isotonic.predict(oof_fusion)

platt = LogisticRegression(penalty=None, solver="lbfgs")
platt.fit(oof_fusion.reshape(-1, 1), y)
cal_oof_platt = platt.predict_proba(oof_fusion.reshape(-1, 1))[:, 1]

calibration_summary = pd.DataFrame([
    {"calibrator": "raw (uncalibrated)", "brier": brier_raw, "AUPRC": auprc_raw},
    {"calibrator": "isotonic", "brier": brier_score_loss(y, cal_oof_iso), "AUPRC": auprc(y, cal_oof_iso)},
    {"calibrator": "platt", "brier": brier_score_loss(y, cal_oof_platt), "AUPRC": auprc(y, cal_oof_platt)},
]).set_index("calibrator")
calibration_summary


C:\Users\aneja\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,brier,AUPRC
calibrator,,
raw (uncalibrated),0.082648,0.516346
isotonic,0.081096,0.504483
platt,0.083576,0.516346


## 6. Final submission

In [15]:
final_test_preds = platt.predict_proba(test_preds_fusion.reshape(-1, 1))[:, 1]

submission = pd.DataFrame({
    "flight_id": tab_test["flight_id"],
    "failure_within_horizon": final_test_preds,
})
submission.to_csv("../submission.csv", index=False)

sample_submission = pd.read_csv("../data/sample_submission.csv")
assert len(submission) == len(sample_submission)
assert set(submission["flight_id"]) == set(sample_submission["flight_id"])
assert submission["failure_within_horizon"].between(0, 1).all()
assert not submission["failure_within_horizon"].isna().any()
print("submission.csv written and verified:", submission.shape)
submission["failure_within_horizon"].describe()


submission.csv written and verified: (7534, 2)


count    7534.000000
mean        0.127532
std         0.163354
min         0.056516
25%         0.064007
50%         0.070093
75%         0.100391
max         0.963105
Name: failure_within_horizon, dtype: float64